# Step 3: Data Cleaning
### Marketing Funnel Analysis: Bank Marketing Campaign Dataset

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/bank-additional-full.csv', sep=';')
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
print("'Unknown' value counts per column:")
for col in df.columns:
    count = (df[col] == 'unknown').sum()
    if count > 0:
        pct = count / len(df) * 100
        print(f"  {col:<20} → {count:>5} unknowns ({pct:.1f}%)")

## Strategy for 'unknown' values
- `job` → keep, tag as 'unknown' (meaningful segment)
- `marital` → keep, tag as 'unknown' (only 0.2%)
- `education` → keep, tag as 'unknown' (meaningful segment)
- `default` → drop column (>20% unknown, not useful)
- `housing` → keep (unknowns are small)
- `loan` → keep (unknowns are small)
- `poutcome` → keep (unknown = no previous campaign, meaningful)

We will NOT drop rows, unknowns are a real customer segment.

In [ ]:
df.drop(columns=['default'], inplace=True)
print("Dropped 'default' column")
print(f"Shape now: {df.shape}")

In [ ]:
df['y_binary'] = (df['y'] == 'yes').astype(int)
print("Target column encoded:")
print(df[['y', 'y_binary']].value_counts())

In [ ]:
# These should be categorical
cat_cols = ['job', 'marital', 'education', 'housing', 
            'loan', 'contact', 'month', 'day_of_week', 
            'poutcome', 'y']

for col in cat_cols:
    df[col] = df[col].astype('category')

print("Categorical columns set:")
print(df.dtypes[df.dtypes == 'category'])

In [ ]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[17, 25, 35, 45, 55, 65, 100],
    labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
)

print("Age group distribution:")
print(df['age_group'].value_counts().sort_index())

In [ ]:
df['campaign_bucket'] = pd.cut(
    df['campaign'],
    bins=[0, 1, 3, 5, 100],
    labels=['1 contact', '2-3 contacts', '4-5 contacts', '6+ contacts']
)

print("Campaign intensity distribution:")
print(df['campaign_bucket'].value_counts().sort_index())

In [ ]:
print("=== CLEANED DATASET SUMMARY ===")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nData types:\n{df.dtypes}")
print(f"\nAny nulls: {df.isnull().sum().sum()}")

In [ ]:
df.to_csv('../data/bank_cleaned.csv', index=False)
print("Cleaned dataset saved to data/bank_cleaned.csv")

## Cleaning Summary
- Dropped `default` column (too many unknowns, low value)
- Encoded `y` → `y_binary` (0/1) for calculations
- Set correct categorical data types
- Created `age_group` column for segmentation
- Created `campaign_bucket` column for funnel intensity analysis
- No rows dropped, all 41,188 records retained
- Dataset ready for funnel metric calculation